# Semana 2 · Sesión 1: Clases y objetos

**Módulo 0**

## Objetivos de la sesión

1. Definir clases propias con atributos, métodos y `__init__`.
2. Distinguir métodos de instancia, `@property`, `@classmethod` y
   `@staticmethod`.
3. Sobrecargar operadores con métodos especiales (`__repr__`, `__str__`,
   `__eq__`, `__add__`).

## Antes de empezar

Esta clase sigue con el mismo entorno de la semana 1 — no hay nada nuevo
que instalar. Si no revisaste el checklist de
[`preparacion.md`](../preparacion/preparacion.md), hazlo ahora.

Vamos a construir una sola clase, `Vector2D`, **por capas**: cada sección
le agrega una capacidad y explica qué problema resuelve. Como es normal al
trabajar en un notebook, redefinimos la clase completa en cada paso; para
que cada celda quepa en pantalla, cada versión se queda solo con lo que
necesita la sección.

La sesión 2 de esta semana retoma esta misma clase para hablar de
inmutabilidad y herencia.

In [ ]:
import math

## De funciones a objetos

Hasta ahora representábamos una fuerza en el plano como una tupla de dos
números:

```python
fuerza = (3.0, 4.0)
magnitud = math.sqrt(fuerza[0]**2 + fuerza[1]**2)
```

Funciona, pero el dato y las operaciones que le corresponden viven
separados: nada impide pasarle esa tupla a una función que espera una
posición, y `fuerza[0]` no dice en ningún lado que sea la componente $x$.

Una **clase** junta las dos cosas —los datos (*atributos*) y lo que se
puede hacer con ellos (*métodos*)— bajo un nombre que dice qué es el
objeto.

## Clases y objetos: `__init__`, atributos y métodos

- La **clase** es el molde; el **objeto** (o *instancia*) es cada ejemplar
  construido con ese molde.
- `__init__` se ejecuta al construir la instancia: recibe los datos y los
  guarda como atributos.
- `self` es la instancia sobre la que se está trabajando. Es el primer
  parámetro de todo método de instancia, y Python lo pasa solo — no lo
  escribes al llamar el método.

In [ ]:
class Vector2D:
    """Vector en el plano, dado por sus componentes cartesianas."""

    def __init__(self, x, y):
        self.x = x   # atributos de instancia: cada vector tiene los suyos
        self.y = y

    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)


v = Vector2D(3.0, 4.0)
v.magnitud()   # Vector2D es el molde; v es un objeto construido con él

## TODO en clase 1

Agrega a `Vector2D` un método `escalar(self, k)` que devuelva **un vector
nuevo** con ambas componentes multiplicadas por $k$:

$$k\,\vec{v} = (k\,v_x,\; k\,v_y)$$

Ojo con la palabra *nuevo*: el método no debe modificar `self`, sino
devolver un `Vector2D(...)` recién construido. Vamos a insistir mucho en
eso hoy.

In [ ]:
# TODO en clase: agrega el método escalar(self, k), que devuelve un Vector2D nuevo
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)

    def escalar(self, k):
        ...

## Depuración en vivo: el error número uno

Antes de seguir, un alto para ver el error que **todo el mundo** comete la
primera semana escribiendo clases. Lee la clase de abajo y, antes de
ejecutarla, predice qué va a pasar:

In [ ]:
class VectorRoto:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def magnitud():   # <-- ¿ves lo que falta?
        return 0.0


try:
    VectorRoto(3.0, 4.0).magnitud()
except TypeError as error:
    print("TypeError:", error)

El mensaje dice *"takes 0 positional arguments but 1 was given"*, y suena
absurdo: no le pasamos ningún argumento. Pero sí lo hicimos — Python pasa
la instancia **automáticamente** como primer argumento a todo método de
instancia. Al escribir `magnitud()` sin `self`, el método declara que no
acepta nada, y esa instancia que Python inserta sobra.

Traducción práctica: cuando veas *"takes 0 positional arguments but 1 was
given"* en una clase, casi siempre te falta `self`.

## Instancia vs. clase; `@property`, `@classmethod`, `@staticmethod`

No todo lo que vive en una clase pertenece a cada instancia:

| Elemento | Recibe | Para qué sirve |
|---|---|---|
| Atributo de instancia (`self.x`) | — | Dato propio de cada objeto |
| Atributo de clase (`dimension = 2`) | — | Dato compartido por todas las instancias |
| Método de instancia | `self` | Opera sobre *este* objeto |
| `@property` | `self` | Método que se **usa como si fuera un atributo**: `v.magnitud`, sin paréntesis |
| `@classmethod` | `cls` | Constructor alternativo: otra forma de fabricar instancias |
| `@staticmethod` | nada | Función relacionada con la clase, que no necesita ni la instancia ni la clase |

`@property` es la que más vamos a usar. Sirve para exponer un valor
*calculado* con la misma sintaxis que un atributo guardado: quien usa el
objeto no necesita saber cuál de las dos cosas es.

In [ ]:
class Vector2D:

    dimension = 2   # atributo de clase: lo comparten todas las instancias

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @property
    def magnitud(self):
        # Se calcula al momento, pero se lee como atributo: v.magnitud
        return math.sqrt(self.x**2 + self.y**2)

    @classmethod
    def desde_polares(cls, r, theta):
        # Constructor alternativo: fabrica un Vector2D a partir de (r, theta)
        return cls(r * math.cos(theta), r * math.sin(theta))

    @staticmethod
    def grados_a_radianes(grados):
        # Ni self ni cls: es una utilidad que vive aquí solo por contexto
        return grados * math.pi / 180


v = Vector2D(3.0, 4.0)
w = Vector2D.desde_polares(1.0, Vector2D.grados_a_radianes(90))

v.magnitud, w.y, Vector2D.dimension, Vector2D.grados_a_radianes(180)

Fíjate en la diferencia de las tres últimas: `magnitud` se lee como
atributo (sin paréntesis) porque es `@property`; `desde_polares` se llama
sobre la **clase** y devuelve una instancia nueva; y `grados_a_radianes`
ni siquiera necesitó una instancia — es una función normal que guardamos
dentro de la clase porque es ahí donde tiene sentido buscarla.

## TODO en clase 2

![Vector y el ángulo θ que forma con el eje x](img/vector-angulo.svg)

Agrega una `@property` llamada `angulo` que devuelva el ángulo del vector
respecto al eje $x$, en radianes.

Usa `math.atan2(self.y, self.x)`, **no** `math.atan(self.y / self.x)`:
`atan2` mira el signo de las dos componentes y por eso acierta el
cuadrante, además de no romperse cuando $v_x = 0$. Compruébalo con
`Vector2D(-1.0, 0.0)`: el ángulo debe ser $\pi$, no $0$.

In [ ]:
# TODO en clase: agrega la @property angulo, usando math.atan2
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @property
    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)

    @property
    def angulo(self):
        ...

## Métodos especiales (*dunder methods*)

Los métodos con doble guion bajo (*double underscore* → *dunder*) son la
forma en que una clase propia se engancha a la sintaxis de Python. No los
llamas tú: los llama el intérprete.

| Escribes | Python llama |
|---|---|
| `repr(v)`, o el resultado de una celda | `v.__repr__()` |
| `print(v)`, `str(v)` | `v.__str__()` (si falta, usa `__repr__`) |
| `v == w` | `v.__eq__(w)` |
| `v + w` | `v.__add__(w)` |
| `v * k` | `v.__mul__(k)` |
| `hash(v)`, `{v: ...}` | `v.__hash__()` |

Sin `__repr__`, el notebook muestra algo como
`<__main__.Vector2D object at 0x7f...>`, que no informa nada. Sin
`__eq__`, `Vector2D(1, 2) == Vector2D(1, 2)` resulta `False`: por defecto
Python compara **identidad** (si son el mismo objeto en memoria), no
contenido.

Convención para `__repr__`: devolver, cuando se pueda, el código que
reconstruye el objeto — `Vector2D(3.0, 4.0)`.

In [ ]:
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented   # deja que Python intente la operación inversa
        return self.x == otro.x and self.y == otro.y

    def __add__(self, otro):
        return Vector2D(self.x + otro.x, self.y + otro.y)


Vector2D(3.0, 4.0) + Vector2D(1.0, 1.0), Vector2D(1, 2) == Vector2D(1, 2)

## `__repr__` y `__str__`: dos públicos distintos

Los dos convierten el objeto en texto, pero no para la misma persona:

- `__repr__` es para **quien programa y depura**. La convención es que
  devuelva, si se puede, el código que reconstruye el objeto.
- `__str__` es para **quien lee la salida**. Puede ser más bonito y menos
  literal.

Si defines solo `__repr__`, Python lo usa también donde tocaría `__str__`
— por eso hasta ahora nos bastó con uno. Al definir los dos, se nota
quién usa cuál:

In [ ]:
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __str__(self):
        return f"({self.x}, {self.y})"


v = Vector2D(3.0, 4.0)

print(v)         # print usa __str__
print(repr(v))   # repr() pide explícitamente __repr__

[v, v]           # ojo: dentro de una lista, Python siempre usa __repr__

## TODO en clase 3

Agrega dos métodos especiales más:

1. `__mul__(self, k)`, para que `v * 3` devuelva el vector escalado — el
   mismo cálculo del método `escalar` del TODO 1, ahora con sintaxis de
   operador.
2. `__sub__(self, otro)`, para que `v - w` funcione.

Los dos devuelven un `Vector2D` **nuevo**. Al terminar, comprueba que
`Vector2D(3.0, 4.0) * 2` da `Vector2D(6.0, 8.0)`.

In [ ]:
# TODO en clase: agrega __mul__ (por un escalar) y __sub__; ambos devuelven un Vector2D nuevo
class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented
        return self.x == otro.x and self.y == otro.y

    def __add__(self, otro):
        return Vector2D(self.x + otro.x, self.y + otro.y)

    def __mul__(self, k):
        ...

    def __sub__(self, otro):
        ...

## Resumen

Hoy construimos una clase propia por capas: atributos y `__init__`,
`@property` para valores calculados, `@classmethod` como constructor
alternativo, `@staticmethod` para utilidades sin estado, y los métodos
especiales que enganchan la clase a la sintaxis de Python (`__repr__`,
`__str__`, `__eq__`, `__add__`).

También vimos el error más común al empezar: olvidar `self` en la firma de
un método.

**Próxima sesión — Semana 2, sesión 2:** inmutabilidad y herencia. Vamos a
retomar este mismo `Vector2D` para ver por qué los objetos matemáticos no
deben poder modificarse, y cómo se construyen jerarquías de clases.